# Operações com Pandas

## Caso: relatório de vendas de uma loja

**objetivos**:

- juntar tabelas com merge;
- calcular indicadores;
- resumir dados com groupby;
- criar uma tabela dinâmica;
- comparar resultados com metas;
- apresentar uma recomendação baseada em dados.

## Situação problema

A empresa registra pedidos em uma tabela, clientes em outra tabela, produtos e lojas em outras tabelas também. A gestão quer um relatório que combine essas informações.

**merge** = serve para responder a seguinte pergunta: "como trazer informações de outra tabela?"

**groupby** = "Como calcular resultados para cada grupo?"

**pivot_table** = Como posso comparar grupos em linhas e colunas?

# Ler arquivos e compreender cada tabela

In [1]:
from google.colab import files
uploaded = files.upload()

print(list(uploaded.keys()))

Saving clientes.csv to clientes.csv
Saving lojas.json to lojas.json
Saving metas_categorias.xlsx to metas_categorias.xlsx
Saving pedidos.csv to pedidos.csv
Saving produtos.csv to produtos.csv
['clientes.csv', 'lojas.json', 'metas_categorias.xlsx', 'pedidos.csv', 'produtos.csv']


In [2]:
import pandas as pd

pedidos = pd.read_csv("pedidos.csv")

clientes = pd.read_csv("clientes.csv")

produtos = pd.read_csv("produtos.csv")

lojas = pd.read_json("lojas.json")

metas = pd.read_excel(
    "metas_categorias.xlsx",
    sheet_name="Metas"
)

In [3]:
display(pedidos.head())
display(clientes.head())
display(produtos.head())
display(lojas.head())
display(metas.head())

,pedido_id,data,cliente_id,produto_id,loja_id,quantidade,status,cupom
0,1001,2026-10-11,2,101,4,3,entregue,NÃO
1,1002,2026-10-03,12,102,4,5,entregue,NÃO
2,1003,2026-10-01,1,102,1,2,cancelado,SIM
3,1004,2026-10-09,4,107,1,4,cancelado,NÃO
4,1005,2026-10-13,1,103,4,4,entregue,NÃO


,cliente_id,cliente,estado,segmento,idade
0,1,Ana Souza,RJ,Recorrente,32
1,2,Bruno Lima,SP,Novo,28
2,3,Carla Dias,MG,Recorrente,41
3,4,Diego Rocha,RJ,Novo,23
4,5,Eva Martins,SP,VIP,37


,produto_id,produto,categoria,preco_unitario,custo_unitario
0,101,Mouse sem fio,Periféricos,89.9,46.0
1,102,Teclado mecânico,Periféricos,249.9,135.0
2,103,Webcam HD,Vídeo,179.9,92.0
3,104,Headset,Áudio,199.9,105.0
4,105,Suporte notebook,Acessórios,119.9,62.0


,loja_id,loja,canal
0,1,Online,E-commerce
1,2,Marketplace,Parceiro
2,3,Loja RJ,Físico
3,4,Loja SP,Físico


,categoria,meta_vendas,meta_margem
0,Periféricos,6500,0.38
1,Vídeo,9000,0.30
2,Áudio,3500,0.35
3,Acessórios,4000,0.40


## Merge

Prepara a nossa base analítica com merges

In [4]:
dados = pedidos.merge(
    clientes,
    on="cliente_id",
    how="left",
    validate="many_to_one"
)

display(dados.head())

,pedido_id,data,cliente_id,produto_id,loja_id,quantidade,status,cupom,cliente,estado,segmento,idade
0,1001,2026-10-11,2,101,4,3,entregue,NÃO,Bruno Lima,SP,Novo,28
1,1002,2026-10-03,12,102,4,5,entregue,NÃO,Marcos Silva,SP,Recorrente,42
2,1003,2026-10-01,1,102,1,2,cancelado,SIM,Ana Souza,RJ,Recorrente,32
3,1004,2026-10-09,4,107,1,4,cancelado,NÃO,Diego Rocha,RJ,Novo,23
4,1005,2026-10-13,1,103,4,4,entregue,NÃO,Ana Souza,RJ,Recorrente,32


In [5]:
# Acrescentando produtos

dados = dados.merge(
    produtos,
    on="produto_id",
    how="left",
    validate="many_to_one"
)

In [6]:
# Acrescentando lojas

dados = dados.merge(
    lojas,
    on="loja_id",
    how="left",
    validate="many_to_one"
)

In [7]:
display(dados.head())

,pedido_id,data,cliente_id,produto_id,loja_id,quantidade,status,cupom,cliente,estado,segmento,idade,produto,categoria,preco_unitario,custo_unitario,loja,canal
0,1001,2026-10-11,2,101,4,3,entregue,NÃO,Bruno Lima,SP,Novo,28,Mouse sem fio,Periféricos,89.9,46.0,Loja SP,Físico
1,1002,2026-10-03,12,102,4,5,entregue,NÃO,Marcos Silva,SP,Recorrente,42,Teclado mecânico,Periféricos,249.9,135.0,Loja SP,Físico
2,1003,2026-10-01,1,102,1,2,cancelado,SIM,Ana Souza,RJ,Recorrente,32,Teclado mecânico,Periféricos,249.9,135.0,Online,E-commerce
3,1004,2026-10-09,4,107,1,4,cancelado,NÃO,Diego Rocha,RJ,Novo,23,Monitor 24,Vídeo,849.9,590.0,Online,E-commerce
4,1005,2026-10-13,1,103,4,4,entregue,NÃO,Ana Souza,RJ,Recorrente,32,Webcam HD,Vídeo,179.9,92.0,Loja SP,Físico


In [8]:
print("Pedidos antes das junções: ", len(pedidos))
print("Registros depois das junções: ", len(dados))

print(
    dados[
    ["cliente", "categoria", "preco_unitario", "loja", "canal"]
    ].isna().sum()
)

Pedidos antes das junções:  60
Registros depois das junções:  60
cliente           0
categoria         0
preco_unitario    0
loja              0
canal             0
dtype: int64


# Criando métricas

In [9]:
# Valor total do pedido

dados["valor_total"] = (dados["quantidade"] * dados["preco_unitario"])

In [10]:
display(dados.head())

,pedido_id,data,cliente_id,produto_id,loja_id,quantidade,status,cupom,cliente,estado,segmento,idade,produto,categoria,preco_unitario,custo_unitario,loja,canal,valor_total
0,1001,2026-10-11,2,101,4,3,entregue,NÃO,Bruno Lima,SP,Novo,28,Mouse sem fio,Periféricos,89.9,46.0,Loja SP,Físico,269.7
1,1002,2026-10-03,12,102,4,5,entregue,NÃO,Marcos Silva,SP,Recorrente,42,Teclado mecânico,Periféricos,249.9,135.0,Loja SP,Físico,1249.5
2,1003,2026-10-01,1,102,1,2,cancelado,SIM,Ana Souza,RJ,Recorrente,32,Teclado mecânico,Periféricos,249.9,135.0,Online,E-commerce,499.8
3,1004,2026-10-09,4,107,1,4,cancelado,NÃO,Diego Rocha,RJ,Novo,23,Monitor 24,Vídeo,849.9,590.0,Online,E-commerce,3399.6
4,1005,2026-10-13,1,103,4,4,entregue,NÃO,Ana Souza,RJ,Recorrente,32,Webcam HD,Vídeo,179.9,92.0,Loja SP,Físico,719.6


In [11]:
# Custo dos produtos do pedido

dados["custo_total"] = (dados["quantidade"]* dados["custo_unitario"])

In [12]:
display(dados.head())

,pedido_id,data,cliente_id,produto_id,loja_id,quantidade,status,cupom,cliente,estado,segmento,idade,produto,categoria,preco_unitario,custo_unitario,loja,canal,valor_total,custo_total
0,1001,2026-10-11,2,101,4,3,entregue,NÃO,Bruno Lima,SP,Novo,28,Mouse sem fio,Periféricos,89.9,46.0,Loja SP,Físico,269.7,138.0
1,1002,2026-10-03,12,102,4,5,entregue,NÃO,Marcos Silva,SP,Recorrente,42,Teclado mecânico,Periféricos,249.9,135.0,Loja SP,Físico,1249.5,675.0
2,1003,2026-10-01,1,102,1,2,cancelado,SIM,Ana Souza,RJ,Recorrente,32,Teclado mecânico,Periféricos,249.9,135.0,Online,E-commerce,499.8,270.0
3,1004,2026-10-09,4,107,1,4,cancelado,NÃO,Diego Rocha,RJ,Novo,23,Monitor 24,Vídeo,849.9,590.0,Online,E-commerce,3399.6,2360.0
4,1005,2026-10-13,1,103,4,4,entregue,NÃO,Ana Souza,RJ,Recorrente,32,Webcam HD,Vídeo,179.9,92.0,Loja SP,Físico,719.6,368.0


In [13]:
# Margem em reais

dados["margem"] = (dados["valor_total"] - dados["custo_total"])

display(dados.head())

,pedido_id,data,cliente_id,produto_id,loja_id,quantidade,status,cupom,cliente,estado,...,idade,produto,categoria,preco_unitario,custo_unitario,loja,canal,valor_total,custo_total,margem
0,1001,2026-10-11,2,101,4,3,entregue,NÃO,Bruno Lima,SP,...,28,Mouse sem fio,Periféricos,89.9,46.0,Loja SP,Físico,269.7,138.0,131.7
1,1002,2026-10-03,12,102,4,5,entregue,NÃO,Marcos Silva,SP,...,42,Teclado mecânico,Periféricos,249.9,135.0,Loja SP,Físico,1249.5,675.0,574.5
2,1003,2026-10-01,1,102,1,2,cancelado,SIM,Ana Souza,RJ,...,32,Teclado mecânico,Periféricos,249.9,135.0,Online,E-commerce,499.8,270.0,229.8
3,1004,2026-10-09,4,107,1,4,cancelado,NÃO,Diego Rocha,RJ,...,23,Monitor 24,Vídeo,849.9,590.0,Online,E-commerce,3399.6,2360.0,1039.6
4,1005,2026-10-13,1,103,4,4,entregue,NÃO,Ana Souza,RJ,...,32,Webcam HD,Vídeo,179.9,92.0,Loja SP,Físico,719.6,368.0,351.6


In [14]:
display(
    dados[
        [
            "pedido_id",
            "categoria",
            "quantidade",
            "valor_total",
            "custo_total",
            "margem",
            "status"
        ]
    ].head()
)

,pedido_id,categoria,quantidade,valor_total,custo_total,margem,status
0,1001,Periféricos,3,269.7,138.0,131.7,entregue
1,1002,Periféricos,5,1249.5,675.0,574.5,entregue
2,1003,Periféricos,2,499.8,270.0,229.8,cancelado
3,1004,Vídeo,4,3399.6,2360.0,1039.6,cancelado
4,1005,Vídeo,4,719.6,368.0,351.6,entregue


| Indicador | Cálculo | Significado |
|---|---|---|
| `valor_total` | Quantidade × preço unitário | Valor dos produtos no pedido |
| `custo_total` | Quantidade × custo unitário | Custo dos produtos no pedido |
| `margem` | Valor total − custo total | Diferença entre o valor da venda e o custo dos produtos |

# Vendas realizadas

Decisão de análise: vamos considerar apenas pedidos entregues

In [18]:
print(dados["status"].value_counts())

status
entregue     36
cancelado    12
pendente     12
Name: count, dtype: int64


In [19]:
vendas = dados.loc[
    dados["status"] == "entregue"
].copy()

print("Total de pedidos: ", len(dados))
print("Pedidos entregues: ", len(vendas))

Total de pedidos:  60
Pedidos entregues:  36


# Groupby

organiza os registros em grupos. Permite separar os valores em cada parte.

Permite calcular indicadores diferentes a partir das mesmas linhas.

In [20]:
faturamento_categoria = (
    vendas # escolhemos a tabela de pedidos entregues
    .groupby("categoria")["valor_total"].sum() # organizamos os registros em grupos
)
display(faturamento_categoria)

,valor_total
categoria,
Acessórios,2238.0
Periféricos,10075.2
Vídeo,11557.3
Áudio,2598.7


In [21]:
resumo_categoria = (
    vendas
    .groupby("categoria", as_index=False)
    .agg(
        faturamento=("valor_total", "sum"),
        margem=("margem", "sum"),
        unidades=("quantidade", "sum"),
        pedidos=("pedido_id", "nunique")
    )
)

resumo_categoria = resumo_categoria.sort_values(
    "faturamento",
    ascending=False
)

display(resumo_categoria.round(2))

,categoria,faturamento,margem,unidades,pedidos
2,Vídeo,11557.3,4093.3,27,8
1,Periféricos,10075.2,4663.2,48,15
3,Áudio,2598.7,1233.7,13,5
0,Acessórios,2238.0,1096.0,20,8


## A categoria com maior faturamento é também a categoria que tem a maior margem em reais?

Não. A categoria que lidera faturamento é Vídeo; Periféricos lidera em margem em reais.

# Por que 48 unidades na categoria Periféricos não significam 48 pedidos?

Um pedido pode conter várias unidades.

## Relação com o trabalho:

um relatório precisa distinguir volume de produtos, quantidade de pedidos e valores financeiros. São indicadores diferentes, mesmo quando vêm das mesmas linhas.

# Agrupar por duas informações: estado e segmento

"Quais combinações de estado e segmento de clientes geraram mais faturamento?"

In [23]:
resumo_estado_segmento = (
    vendas
    .groupby(["estado", "segmento"], as_index=False)
    .agg(
        faturamento=("valor_total", "sum"),
        pedidos=("pedido_id", "nunique")

    ).sort_values("faturamento", ascending=False)
)

display(resumo_estado_segmento.round(2))

,estado,segmento,faturamento,pedidos
5,RJ,Novo,10398.0,5
6,RJ,Recorrente,3348.4,5
0,BA,Recorrente,3168.4,5
3,PE,Recorrente,2768.3,5
9,SP,Recorrente,2369.2,4
2,MG,VIP,1719.0,5
4,PR,Novo,1079.2,2
1,MG,Recorrente,629.4,2
10,SP,VIP,599.7,1
8,SP,Novo,269.7,1


# Pivot_table

Cria uma tabela dinâmica. Equivalente a tabela dinâmica do Excel.

Resumir os dados em uma tabela cruzada, com uma coluna da tabela original virando as linhas, outra virando as colunas, e os valores agregados(soma, média, contagem) no meio.

## Quanto cada categoria vendeu em cada canal?

In [25]:
tabela_canais = pd.pivot_table(
    vendas,
    values="valor_total", #Trabalha com o valor dos pedidos
    index="categoria", #colocar categorias nas linhas
    columns="canal", #colocar canais nas colunas
    aggfunc="sum", # somar os valores de cada combinação
    fill_value=0, #Mostrar zero nas combinações sem registro
    margins=True, #Acrescentar totais
    margins_name="Total" #Dar o nome "Total" à linha e a coluna de totais
)

display(tabela_canais.round(2))

canal,E-commerce,Físico,Parceiro,Total
categoria,,,,
Acessórios,599.4,799.2,839.4,2238.0
Periféricos,7466.5,2608.7,0.0,10075.2
Vídeo,4969.1,6588.2,0.0,11557.3
Áudio,799.6,1799.1,0.0,2598.7
Total,13834.6,11795.2,839.4,26469.2


- Qual canal teve maior faturamento?
**E-commerce**

- Em qual canal Vídeo teve maior faturamento? **Físico**

- O zero em Áudio/Parceiro prova que ninguém quer comprar áudio nesse canal? **Não. Só mostra que não há vendas entregues dessa combinação nesta base.**

# Comparar os resultados com metas

Saber quanto vendemos e se esse resultado atingiu o objetivo.

Vamos juntar o resultado calculado com a meta de cada categoria.

In [26]:
comparativo = resumo_categoria.merge(
    metas,
    on="categoria",
    how="left",
    validate="one_to_one"
)

In [29]:
# Criando os indicadores

comparativo["atingiu_meta"] = (
    comparativo["faturamento"] >= comparativo["meta_vendas"]
)

comparativo["percentual_meta"] = (
    comparativo["faturamento"]
    / comparativo["meta_vendas"]
    * 100
)

comparativo["diferenca_meta"] = (
    comparativo["faturamento"] - comparativo["meta_vendas"]
)

display(
    comparativo[
        [
            "categoria",
            "faturamento",
            "meta_vendas",
            "percentual_meta",
            "diferenca_meta",
            "atingiu_meta"
        ]
    ].round(2)
)

,categoria,faturamento,meta_vendas,percentual_meta,diferenca_meta,atingiu_meta
0,Vídeo,11557.3,9000,128.41,2557.3,True
1,Periféricos,10075.2,6500,155.00,3575.2,True
2,Áudio,2598.7,3500,74.25,-901.3,False
3,Acessórios,2238.0,4000,55.95,-1762.0,False


In [30]:
# Meta_margem

comparativo["taxa_margem"] = (
    comparativo["margem"] / comparativo["faturamento"]
)
comparativo["atingiu_meta_margem"] = (
    comparativo["taxa_margem"] >= comparativo["meta_margem"]
)

display(comparativo)

,categoria,faturamento,margem,unidades,pedidos,meta_vendas,meta_margem,atingiu_meta,diferenca_meta,percentual_meta,taxa_margem,atingiu_meta_margem
0,Vídeo,11557.3,4093.3,27,8,9000,0.30,True,2557.3,128.414444,0.354174,True
1,Periféricos,10075.2,4663.2,48,15,6500,0.38,True,3575.2,155.003077,0.462839,True
2,Áudio,2598.7,1233.7,13,5,3500,0.35,False,-901.3,74.248571,0.474737,True
3,Acessórios,2238.0,1096.0,20,8,4000,0.40,False,-1762.0,55.950000,0.489723,True


**margem percentual** e **volume de vendas** medem coisas diferentes.

**Vendas(volume ou faturamento)**: Quanto foi vendido no total. Responde: "estamos vendendo o suficiente?"

**Margem percentual**: Responde: "Cada venda está sendo lucrativa?"

In [32]:
# Qual canal teve maior faturamento

resumo_canal = (
    vendas
    .groupby("canal", as_index=False)
    .agg(faturamento=("valor_total", "sum"))
    .sort_values("faturamento", ascending=False)
)
display(resumo_canal.round(2))

,canal,faturamento
0,E-commerce,13834.6
1,Físico,11795.2
2,Parceiro,839.4


# Mini-case de análise

A gestão quer saber:

1. Qual categoria teve maior faturamento?

**Resposta**: Vídeo: R$ 11.557,30



2. Qual canal teve maior faturamento?

**Resposta**: E-commerce: R$ 13.834,60

3. Quais categorias atingiram a meta de vendas?

**Resposta**: Vídeo e Periféricos

4. Qual categoria merece uma investigação inicial? Justificando o porquê.

**Resposta**: Acessórios, com 55,95% da meta e diferença de R$ 1762,00 mesmo tendo a maior margem percentual de lucro. Recomendamos investigar disponibilidaden dos produtos, exposição, desempenho por canal, preços acessíveis, tudo isso antes de uma ação comercial.

In [33]:
comparativo.to_csv(
    "comparativo_categorias.csv",
    index=False
)